# Run Neuron Proofreader

Demonstrates how to use the `neuron-proofreader` library to correct split errors
in UNet-predicted neuron fragments. Uses the same brain dataset loaded by
`load_skeletons.ipynb` / `load_skeletons_from_cache.ipynb`.

**Pipeline overview:**
1. Load fragment skeletons into a `ProposalGraph`
2. Generate proposals (candidate reconnections between nearby fragment endpoints)
3. Extract features (skeleton geometry + 3D image patches)
4. Run GNN inference to accept/reject proposals
5. Merge accepted proposals → corrected skeleton

**Note:** Step 4 requires a trained model checkpoint. This notebook shows the full
pipeline structure; if no checkpoint is available, it demonstrates steps 1-3
(data preparation) and ground-truth label generation for training.

## Setup

In [1]:
import json
import numpy as np
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

from neuron_proofreader.config import Config, MLConfig, ProposalGraphConfig
from neuron_proofreader.proposal_graph import ProposalGraph
from neuron_proofreader.skeleton_graph import SkeletonGraph
from neuron_proofreader.split_proofreading.split_feature_extraction import FeaturePipeline
from neuron_proofreader.split_proofreading.proposal_generation import ProposalGenerator
from neuron_proofreader.split_proofreading.groundtruth_generation import run as generate_gt_labels

## Section 1: Define Paths and Configuration

Same data sources as `load_skeletons.ipynb`.

In [2]:
# Brain and segmentation identifiers
brain_id = "794495"
segmentation_id = "raw.unet_449_splits_and_merges_900000"

# GCS paths
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"
gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
segmentation_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

# Image path (ExaSPIM raw fluorescence)
with open("../configs/exaspim_image_prefixes.json") as f:
    img_prefixes = json.load(f)
img_path = img_prefixes[brain_id]

print(f"Fragments: {fragments_path}")
print(f"Ground truth: {gt_path}")
print(f"Segmentation: {segmentation_path}")
print(f"Image: {img_path}")

Fragments: gs://allen-nd-goog/from_google/794495/whole_brain/raw.unet_449_splits_and_merges_900000/swcs
Ground truth: gs://allen-nd-goog/ground_truth_tracings/794495/voxel
Segmentation: gs://allen-nd-goog/from_google/794495/whole_brain/raw.unet_449_splits_and_merges_900000/
Image: s3://aind-open-data/exaSPIM_794495_2026-01-21_14-25-07_processed_2026-01-29_22-02-08/fusion/fused.zarr/


In [3]:
# Configuration for the proofreading pipeline.
# These parameters control graph construction and proposal generation.

graph_config = ProposalGraphConfig(
    anisotropy=(0.748, 0.748, 1.0),   # ExaSPIM voxel size in microns (x, y, z)
    min_cable_length=40.0,             # Drop fragments shorter than this (microns)
    node_spacing=1.0,                  # Resample spacing between nodes (microns)
    prune_depth=24.0,                  # Prune short terminal branches (microns)
    max_proposals_per_leaf=3,          # Max connection candidates per leaf node
    allow_nonleaf_proposals=False,     # Only propose from leaf (endpoint) nodes
    remove_high_risk_merges=False,
    verbose=True,
)

ml_config = MLConfig(
    batch_size=64,
    brightness_clip=400,               # Clip image intensity for normalization
    device="cuda",                     # Use GPU for inference
    patch_shape=(96, 96, 96),           # 3D image patch around each proposal
    transform=False,                   # No augmentation at inference time
)

config = Config(graph_config, ml_config)
print("Graph config:", graph_config)
print("ML config:", ml_config)

Graph config: ProposalGraphConfig(allow_nonleaf_proposals=False, anisotropy=(0.748, 0.748, 1.0), max_proposals_per_leaf=3, min_cable_length=40.0, node_spacing=1.0, proposals_per_leaf=3, prune_depth=24.0, remove_doubles=True, remove_high_risk_merges=False, trim_endpoints_bool=True, verbose=True)
ML config: MLConfig(batch_size=64, brightness_clip=400, device='cuda', model_name=None, patch_shape=(96, 96, 96), transform=False)


## Section 2: Build the ProposalGraph

Load fragment skeletons from SWC files into a `ProposalGraph`. This is the same
data as `dataset.fragments_graph` in `load_skeletons_from_cache.ipynb`, but
loaded through `neuron-proofreader`'s own graph class which adds proposal tracking.

In [ ]:
# Build the ProposalGraph from fragment SWC files
pred_graph = ProposalGraph(
    anisotropy=graph_config.anisotropy,
    max_proposals_per_leaf=graph_config.max_proposals_per_leaf,
    min_cable_length=graph_config.min_cable_length,
    node_spacing=graph_config.node_spacing,
    prune_depth=graph_config.prune_depth,
    remove_high_risk_merges=graph_config.remove_high_risk_merges,
    verbose=graph_config.verbose,
)

pred_graph.load(fragments_path)

print(f"Fragments loaded:")
print(f"  Nodes: {pred_graph.number_of_nodes():,}")
print(f"  Edges: {pred_graph.number_of_edges():,}")
print(f"  Components: {len(set(pred_graph.node_component_id)):,}")

Load Graphs:  90%|████████▉ | 434877/484546 [16:48<02:57, 279.46it/s]

## Section 3: Generate Proposals

For each leaf node (fragment endpoint), search for nearby nodes from *other*
fragments. Each nearby pair becomes a "proposal" — a candidate reconnection
that the model will score as accept/reject.

The `search_radius` controls how far to look (in microns). Larger radius =
more proposals but slower. Typical values: 10-30 microns.

In [ ]:
search_radius = 20.0  # microns

pred_graph.generate_proposals(
    search_radius=search_radius,
    allow_nonleaf_proposals=graph_config.allow_nonleaf_proposals,
)

n_proposals = pred_graph.n_proposals()
print(f"Proposals generated: {n_proposals:,}")
print(f"Search radius: {search_radius} microns")
print(f"Max proposals per leaf: {graph_config.max_proposals_per_leaf}")

## Section 4: Generate Ground-Truth Labels (for training)

If ground-truth skeletons are available, we can label each proposal as
accept (both endpoints align to the same GT neuron) or reject. This is
used to train the GNN model.

A proposal is accepted if:
- Both fragment endpoints project onto the same GT skeleton (< 7 microns)
- The connection is structurally consistent with GT topology
- The proposal length approximately matches the GT path length

In [ ]:
# Load ground-truth skeletons
gt_graph = SkeletonGraph(
    anisotropy=graph_config.anisotropy,
    min_cable_length=0,
    node_spacing=graph_config.node_spacing,
    verbose=True,
)
gt_graph.load(gt_path)

print(f"Ground truth loaded:")
print(f"  Nodes: {gt_graph.number_of_nodes():,}")
print(f"  Edges: {gt_graph.number_of_edges():,}")
print(f"  Components (neurons): {len(set(gt_graph.node_component_id)):,}")

In [ ]:
# Generate accept/reject labels for each proposal
gt_accepts = generate_gt_labels(gt_graph, pred_graph)

n_accepts = len(gt_accepts)
n_rejects = n_proposals - n_accepts

print(f"Ground-truth labels:")
print(f"  Accept: {n_accepts:,} ({100*n_accepts/max(n_proposals,1):.1f}%)")
print(f"  Reject: {n_rejects:,} ({100*n_rejects/max(n_proposals,1):.1f}%)")

## Section 5: Feature Extraction

For each proposal, the `FeaturePipeline` extracts:
- **Skeleton features**: node degree, radius, number of proposals per node
- **Image features**: 3D image patch (96x96x96) centered on the proposal midpoint

These are packed into a `HeteroGraphData` object (PyTorch Geometric) for the GNN.

In [ ]:
# Initialize the feature extraction pipeline
feature_pipeline = FeaturePipeline(
    graph=pred_graph,
    img_path=img_path,
    brightness_clip=ml_config.brightness_clip,
    patch_shape=ml_config.patch_shape,
)

print(f"FeaturePipeline initialized")
print(f"  Image path: {img_path}")
print(f"  Patch shape: {ml_config.patch_shape}")
print(f"  Brightness clip: {ml_config.brightness_clip}")

## Section 6: Train a Model (bounded by epochs or steps)

This section trains `VisionHGAT` using `neuron-proofreader`'s own
`FragmentsDatasetCollection` and `Trainer`. Set `max_epochs` and/or
`max_steps` to control how long training runs.

In [ ]:
import torch
from torch.utils.data import DataLoader
from neuron_proofreader.machine_learning.gnn_models import VisionHGAT
from neuron_proofreader.machine_learning.train import Trainer
from neuron_proofreader.split_proofreading.split_datasets import FragmentsDatasetCollection

# Bounded training controls. Set max_steps for a quick smoke run,
# or leave it as None to train complete epochs.
max_epochs = 5
max_steps = None  # e.g., 200

# Use CUDA when available, otherwise fall back to CPU.
ml_config.device = "cuda" if torch.cuda.is_available() else "cpu"
config = Config(graph_config, ml_config)

train_collection = FragmentsDatasetCollection(shuffle=True)
train_collection.add_dataset(
    key=brain_id,
    fragments_path=fragments_path,
    img_path=img_path,
    config=config,
    gt_path=gt_path,
)
train_collection.generate_proposals(search_radius)

# The collection dataset yields model-ready (inputs, targets), so keep
# DataLoader batch_size=None to avoid collating already-batched subgraphs.
train_loader = DataLoader(train_collection, batch_size=None, num_workers=0)
val_loader = DataLoader(train_collection, batch_size=None, num_workers=0)

model = VisionHGAT(patch_shape=ml_config.patch_shape)
trainer = Trainer(
    model=model,
    model_name="VisionHGAT",
    output_dir="training_output",
    device=ml_config.device,
    lr=1e-3,
    max_epochs=max_epochs,
    max_steps=max_steps,
)

trainer.run(train_loader, val_loader)
if trainer.global_step == 0:
    raise RuntimeError("No training steps ran; check proposal generation and GT labels.")

trained_model_path = trainer.latest_checkpoint_path
if trained_model_path is None:
    trained_model_path = trainer.save_model("final")

print(f"Training complete. Checkpoint: {trained_model_path}")
print(f"Training log dir: {trainer.log_dir}")

## Section 7: Run Inference with the Trained Model

The `InferencePipeline` loads the fragments without GT labels, scores proposals
with the trained checkpoint, progressively merges accepted proposals, and saves
the proofreading result.

In [ ]:
from neuron_proofreader.split_proofreading.split_inference import InferencePipeline

# To reuse an existing checkpoint instead of Section 6, set external_model_path.
external_model_path = None
checkpoint_path = external_model_path or globals().get("trained_model_path")
if checkpoint_path is None:
    raise RuntimeError("Run Section 6 first or set external_model_path to a .pth checkpoint.")

inference_model = VisionHGAT(patch_shape=ml_config.patch_shape)
inference_model.load_state_dict(torch.load(checkpoint_path, map_location=ml_config.device))
inference_model.to(ml_config.device)
inference_model.eval()

pipeline = InferencePipeline(
    fragments_path=fragments_path,
    img_path=img_path,
    output_dir="proofreader_output",
    model=inference_model,
    config=config,
)

pipeline(
    search_radius=search_radius,
    min_threshold=0.75,
    dt=0.05,
    removal_threshold=0.3,
)

print("Inference complete")
print(f"  Accepted proposals: {len(pipeline.dataset.accepts):,}")
print("  Output saved to: proofreader_output/")

In [ ]:
from pathlib import Path

proofreader_output = Path("proofreader_output")
print("Proofreading artifacts:")
for rel_path in [
    "corrected_swcs/swcs.zip",
    "connections.txt",
    "runtimes.txt",
    "metadata_graph.json",
    "metadata_ml.json",
]:
    path = proofreader_output / rel_path
    print(f"  {rel_path}: {'OK' if path.exists() else 'missing'}")

# Run before/after skeleton metrics to quantify which errors were corrected.
# This can be slow because it re-reads GT SWCs, fragment SWCs, and the label volume.
import sys
sys.path.insert(0, "../scripts")
from proofreading_stats import evaluate_before_after, print_comparison

metrics_output_dir = f"../metrics_out/{brain_id}/{segmentation_id}/proofreader_before_after"
comparison = evaluate_before_after(
    gt_path=gt_path,
    segmentation_path=segmentation_path,
    original_fragments_path=fragments_path,
    proofreader_output_dir="proofreader_output",
    metrics_output_dir=metrics_output_dir,
    anisotropy=graph_config.anisotropy,
    use_anisotropy=False,
    save_merges=True,
    verbose=True,
)
print_comparison(comparison)
print(f"Before/after metric reports saved to: {metrics_output_dir}")

## Alternative: Use Cached BrainDataset

If you already have a `BrainDataset` cached from `load_skeletons_from_cache.ipynb`,
you can reuse its file paths directly instead of re-specifying them.

In [ ]:
from agentic_neuron_proofreader.data_modules.datasets import BrainDataset

# Load the cached dataset (fast — no SWC re-reading)
cache_path = f"../dataset_cache_{brain_id}.pkl"
if os.path.exists(cache_path):
    dataset = BrainDataset.load_from_cache(cache_path)

    # The cached dataset holds the same data that neuron-proofreader needs.
    # Key attributes you can reuse:
    print("Cached BrainDataset loaded")
    print(f"  fragments_graph nodes: {dataset.fragments_graph.number_of_nodes():,}")
    print(f"  gt_graph nodes: {dataset.gt_graph.number_of_nodes():,}")
    print(f"  img_path: {dataset.img_path}")
    print()
    print("To use with neuron-proofreader, pass the same file paths:")
    print(f"  fragments_path = '{fragments_path}'")
    print(f"  gt_path = '{gt_path}'")
    print(f"  img_path = '{dataset.img_path}'")
else:
    print(f"Cache not found: {cache_path}")
    print("Run load_skeletons.ipynb first to generate the cache.")